## Запуск Flask из Jupyter Notebook

**Локальный тест из репозитория:** ядро с `.venv`, браузер на `127.0.0.1` — **Docker не используется**. Сервер Flask поднимается **отдельным процессом** (`python -m recorder.dev_server_main`), а не потоком внутри Jupyter — так стабильнее старт/стоп и порт.

Если в UI **0 KB** при живой записи: по умолчанию stderr FFmpeg уходит в «чёрную дыру», чтобы не блокировать пайп на macOS. Хвост ошибок в JSON — только если в `.env` задать **`RECORDING_FFMPEG_STDERR_PIPE=1`** (может снова дать 0 B на шумных HLS).

### Важно (Jupyter и «сломанный» kernel)

Приложение при старте **отключает перехват SIGINT/SIGTERM для записи**, если вы в **IPython/Jupyter** (иначе обработчик из `shutdown_hooks` вызывает `sys.exit` и **роняет ядро** при прерывании ячейки или при некоторых действиях IDE). В обычном терминале поведение не меняется. Чтобы в Jupyter **включить** хуки завершения, задайте в `.env`: `RECORDING_SHUTDOWN_HOOKS=1` (на свой риск).

### Порядок действий

1. Откройте папку **`ffmpeg-stream-rec`** как корень workspace в Cursor и ядро **`Python (ffmpeg-stream-rec .venv)`** (интерпретатор **`…/ffmpeg-stream-rec/.venv/bin/python`**). Иначе Flask/FFmpeg могут быть «не те», что в проекте.
2. **Restart Kernel** при странных симптомах, затем **«Запуск сервера»**. Перед новым стартом при необходимости выполните **«Остановка сервера»** — она завершит дочерний процесс и удалит `notebooks/.dev_server_state.json`.
3. Если в браузере «не работает», откройте **ровно тот URL**, что напечатан (при занятом 5000 порт будет другой, например **50266**).
4. Опционально: ячейка **«Проверка как у CI»** — та же запись, что в автотесте; если там **OK**, а в браузере нет — открыт не тот хост/порт.
5. **«Остановка сервера»** — корректно закрыть Werkzeug в этом ядре.
6. **Перезапуск ядра** — полный сброс потоков (меню Notebook).

### Если браузер не открылся

Перейдите вручную по адресу из вывода ячейки (обычно http://127.0.0.1:5000 ).

### Занят порт

Раньше Werkzeug при занятом порте вызывал `sys.exit(1)` и ронял ячейку; теперь ячейка сама пробует следующий свободный порт. При желании задайте `export FLASK_RUN_PORT=5001` или поменяйте `PORT` в ячейке.

### Запуск сервера

In [ ]:
import json
import os
import signal
import socket
import subprocess
import sys
import time
import webbrowser
from pathlib import Path
from urllib.error import URLError
from urllib.request import urlopen

from IPython.display import HTML, display

# Отдельный процесс Werkzeug — не daemon-thread в Jupyter (меньше сюрпризов с портами/стопом).
_ffmpeg_rec_dev = globals().setdefault(
    "_FFMPEG_STREAM_REC_DEV_SERVER_STATE",
    {"proc": None},
)


def find_repo_root() -> Path:
    cwd = Path.cwd().resolve()
    for base in [cwd, *cwd.parents]:
        if (base / "recorder" / "__init__.py").is_file():
            return base
    raise FileNotFoundError(
        "Не найден пакет recorder/. Откройте репозиторий ffmpeg-stream-rec как папку workspace."
    )


ROOT = find_repo_root()
os.chdir(ROOT)
_rs = str(ROOT)
if _rs not in sys.path:
    sys.path.insert(0, _rs)

HOST = os.environ.get("FLASK_RUN_HOST", "127.0.0.1")
PORT = int(os.environ.get("FLASK_RUN_PORT", "5000"))
OPEN_BROWSER = True
STATE_FILE = ROOT / "notebooks" / ".dev_server_state.json"

print("Рабочая директория:", ROOT)
print("Python:", sys.executable)

_venv_py = ROOT / ".venv" / "bin" / "python"
if _venv_py.is_file():
    try:
        if Path(sys.executable).resolve() != _venv_py.resolve():
            print(
                "\n⚠ Ядро не из .venv репозитория.\n"
                f"   Сейчас:  {sys.executable}\n"
                f"   Выберите: {_venv_py}\n"
            )
    except OSError:
        pass


def _read_state() -> dict | None:
    if not STATE_FILE.is_file():
        return None
    try:
        return json.loads(STATE_FILE.read_text(encoding="utf-8"))
    except (OSError, json.JSONDecodeError):
        return None


def _kill_pid(pid: int) -> None:
    if pid <= 0:
        return
    try:
        os.kill(pid, signal.SIGTERM)
    except ProcessLookupError:
        return
    except PermissionError:
        return
    t0 = time.monotonic()
    while time.monotonic() - t0 < 8.0:
        try:
            os.kill(pid, 0)
        except ProcessLookupError:
            return
        except OSError:
            return
        time.sleep(0.15)
    try:
        os.kill(pid, signal.SIGKILL)
    except OSError:
        pass


def stop_dev_server() -> None:
    proc = _ffmpeg_rec_dev.get("proc")
    if proc is not None:
        if proc.poll() is None:
            proc.terminate()
            try:
                proc.wait(timeout=12)
            except subprocess.TimeoutExpired:
                proc.kill()
        _ffmpeg_rec_dev["proc"] = None

    st = _read_state()
    if st:
        _kill_pid(int(st.get("pid") or 0))
    try:
        STATE_FILE.unlink()
    except OSError:
        pass
    print("Локальный dev-сервер остановлен (процесс + state-файл).")


def start_dev_server(*, open_browser: bool = True) -> None:
    client_host = "127.0.0.1" if HOST in ("0.0.0.0", "::") else HOST

    proc = _ffmpeg_rec_dev.get("proc")
    if proc is not None and proc.poll() is None:
        st = _read_state() or {}
        url = (st.get("url") or "").rstrip("/") + "/"
        if not url.startswith("http"):
            url = f"http://{client_host}:{PORT}/"
        print(f"Сервер уже запущен (этот kernel) → {url}")
        os.environ["_NB_DEV_SERVER_BASE"] = url.rstrip("/")
        if open_browser:
            webbrowser.open(url)
        display(HTML(f'<p><a href="{url}" target="_blank"><b>{url}</b></a></p>'))
        return

    # Снять зависший state / процесс не из этого kernel
    st0 = _read_state()
    if st0:
        _kill_pid(int(st0.get("pid") or 0))
        try:
            STATE_FILE.unlink()
        except OSError:
            pass

    exe = ROOT / ".venv" / "bin" / "python"
    if not exe.is_file():
        exe = Path(sys.executable)

    env = os.environ.copy()
    env["RECORDINGS_ROOT"] = str((ROOT / "recordings").resolve())
    env["STATE_DB_PATH"] = str((ROOT / "data" / "state.sqlite3").resolve())
    env["LOG_DIR"] = str((ROOT / "logs").resolve())
    env["RECORDING_FFMPEG_STDERR_PIPE"] = "0"
    env["RECORDING_SHUTDOWN_HOOKS"] = "0"
    env["FLASK_RUN_HOST"] = HOST
    env["FLASK_RUN_PORT"] = str(PORT)

    try:
        STATE_FILE.unlink()
    except OSError:
        pass

    proc = subprocess.Popen(
        [str(exe), "-m", "recorder.dev_server_main"],
        cwd=str(ROOT),
        env=env,
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL,
    )
    _ffmpeg_rec_dev["proc"] = proc

    url = None
    for _ in range(150):
        time.sleep(0.1)
        if proc.poll() is not None:
            print(
                "Процесс dev_server_main завершился с кодом",
                proc.returncode,
                "— см. logs/app.log в репозитории.",
            )
            _ffmpeg_rec_dev["proc"] = None
            return
        st = _read_state()
        if st and st.get("url"):
            url = st["url"].rstrip("/") + "/"
            break

    if not url:
        print("Не дождались notebooks/.dev_server_state.json — проверьте логи.")
        return

    os.environ["_NB_DEV_SERVER_BASE"] = url.rstrip("/")
    try:
        with urlopen(url, timeout=5) as resp:
            print(f"Проверка: главная страница отвечает HTTP {resp.status}")
    except URLError as e:
        print("Пока не удалось подключиться:", e)

    print(f"\nПриложение: {url}\n")
    display(HTML(f'<p><a href="{url}" target="_blank"><b>Открыть в браузере</b></a></p>'))

    if open_browser:
        opened = webbrowser.open(url)
        print(
            "Браузер открыт автоматически."
            if opened
            else "Откройте ссылку выше вручную (важно: порт может быть не 5000)."
        )


start_dev_server(open_browser=OPEN_BROWSER)


Рабочая директория: /Users/victorpolyakov/dev/ffmpeg-stream-rec
Python: /Users/victorpolyakov/dev/ffmpeg-stream-rec/.venv/bin/python


[2026-05-04 22:39:30,012] INFO in __init__: Приложение создано: RECORDINGS_ROOT=/Users/victorpolyakov/dev/ffmpeg-stream-rec/recordings STATE_DB=/Users/victorpolyakov/dev/ffmpeg-stream-rec/data/state.sqlite3


Порт 5000 занят (часто AirPlay на macOS) → слушаем 50266. Или задайте FLASK_RUN_PORT.


127.0.0.1 - - [04/May/2026 22:39:30] "GET / HTTP/1.1" 200 -


Проверка: главная страница отвечает HTTP 200

Приложение: http://127.0.0.1:50266/



Браузер открыт автоматически.


127.0.0.1 - - [04/May/2026 22:39:31] "GET / HTTP/1.1" 200 -
127.0.0.1 - - [04/May/2026 22:39:31] "GET /static/style.css HTTP/1.1" 200 -
127.0.0.1 - - [04/May/2026 22:39:31] "GET /static/live-status.js HTTP/1.1" 200 -
127.0.0.1 - - [04/May/2026 22:39:31] "GET /static/history-modal.js HTTP/1.1" 200 -
127.0.0.1 - - [04/May/2026 22:39:31] "GET /record/status HTTP/1.1" 200 -
127.0.0.1 - - [04/May/2026 22:39:31] "GET /favicon.ico HTTP/1.1" 404 -
127.0.0.1 - - [04/May/2026 22:39:32] "GET /record/status HTTP/1.1" 200 -
127.0.0.1 - - [04/May/2026 22:39:34] "GET /record/status HTTP/1.1" 200 -
127.0.0.1 - - [04/May/2026 22:39:35] "GET /record/status HTTP/1.1" 200 -
127.0.0.1 - - [04/May/2026 22:39:37] "GET /record/status HTTP/1.1" 200 -
127.0.0.1 - - [04/May/2026 22:39:38] "GET /record/status HTTP/1.1" 200 -
127.0.0.1 - - [04/May/2026 22:39:40] "GET /record/status HTTP/1.1" 200 -
127.0.0.1 - - [04/May/2026 22:39:41] "GET /record/status HTTP/1.1" 200 -
127.0.0.1 - - [04/May/2026 22:39:43] "GET /re

### Остановка сервера

Корректно закрывает HTTP-сервер Werkzeug в этом ядре. Активные записи FFmpeg при этом **не останавливает** — ими управляет веб-приложение или отдельные процессы.

In [76]:
stop_dev_server()

Локальный сервер остановлен (порт должен быть свободен).


### Перезапуск ядра Jupyter

Полностью перезапускает kernel (как **Notebook: Restart Kernel**): обнуляются переменные, останавливаются потоки. После этого снова выполните ячейку **«Запуск сервера»**.

Используйте палитру команд: **Jupyter: Restart Kernel** (из кода kernel принудительно не перезапускаем — в IDE это ошибочно показывается как crash).

In [56]:
# Не вызываем kernel.do_shutdown из кода — в VS Code/Cursor это часто
# показывается как «The Kernel crashed». Перезапуск ядра:
# меню Notebook / Command Palette → «Restart Kernel» (или кнопка ↻ у notebook).
print(
    "Перезапуск ядра: Command Palette → «Jupyter: Restart Kernel» "
    "или меню notebook → Restart."
)

Перезапуск ядра: Command Palette → «Jupyter: Restart Kernel» или меню notebook → Restart.


### Проверка «как у CI» (опционально)

После успешного **«Запуск сервера»** выполните ячейку ниже: она дернёт тот же хост, что записан в `os.environ["_NB_DEV_SERVER_BASE"]`, начнёт короткую запись с публичного HLS и выведет размер. Если здесь **OK**, а в браузере нет — вы открыли **не тот URL/порт** (см. вывод ячейки запуска).

In [ ]:
import json
import os
import time
import urllib.parse
import urllib.request
from pathlib import Path

base = (os.environ.get("_NB_DEV_SERVER_BASE") or "").rstrip("/")
if not base:
    print("Сначала выполните ячейку «Запуск сервера» (успешный старт задаёт _NB_DEV_SERVER_BASE).")
else:
    print("Проверка записи на:", base)

    def http_req(method: str, path: str, form: dict | None = None):
        u = base + path
        if form is not None:
            body = urllib.parse.urlencode(form).encode("utf-8")
            req = urllib.request.Request(u, data=body, method=method)
            req.add_header("Content-Type", "application/x-www-form-urlencoded")
        else:
            req = urllib.request.Request(u, method=method)
        return urllib.request.urlopen(req, timeout=120)

    http_req(
        "POST",
        "/record/start",
        {
            "url": "https://test-streams.mux.dev/x36xhzz/x36xhzz.m3u8",
            "basename": "nb_selftest",
            "storage_mode": "1",
        },
    )
    sid = None
    mx = 0
    for _ in range(20):
        time.sleep(0.5)
        with http_req("GET", "/record/status") as r:
            data = json.loads(r.read().decode())
        for s in data.get("sessions", []):
            if s.get("basename") == "nb_selftest" and s.get("status") == "recording":
                sid = s["id"]
                mx = max(mx, int(s.get("current_file_bytes") or 0))
                break
        if sid and mx > 8000:
            break
    print("session", sid, "max_bytes_while_recording", mx)
    if sid:
        http_req("POST", f"/record/stop/{sid}", None)
    time.sleep(2.0)
    cwd = Path.cwd().resolve()
    root = next(
        (b for b in [cwd, *cwd.parents] if (b / "recorder" / "__init__.py").is_file()),
        None,
    )
    if root is None:
        raise SystemExit("Не найден корень репозитория (recorder/__init__.py)")
    hits = list((root / "recordings").glob("nb_selftest*.mp4"))
    sz = max((p.stat().st_size for p in hits), default=0)
    print("files", [(str(p.name), p.stat().st_size) for p in hits])
    ok = mx > 8000 and sz > 2000
    print("SELFTEST", "OK" if ok else "FAIL")
